# 04 — Ridge Regression Training

Dieses Notebook trainiert **Ridge-Regressionsmodelle** auf allen 5 Datenvarianten.

## Was ist Ridge Regression?
Ridge Regression ist eine **lineare Regression mit L2-Regularisierung**:

$$\min_w \|Xw - y\|^2 + \alpha \|w\|^2$$

Der Term $\alpha \|w\|^2$ bestraft grosse Koeffizienten. Das verhindert Overfitting und macht das Modell robuster gegenueber Multikollinearitaet.

## Warum ist Ridge das beste Modell?
Ueberraschendes Ergebnis: Ridge dominiert die Cross-Gemeinde-Evaluation (4 von 5 Top-Plaetzen). Gruende:
1. **Keine Overfitting-Gefahr**: Lineares Modell kann keine komplexen gemeinde-spezifischen Muster auswendig lernen
2. **Label-Encoding-Robustheit**: Ridge behandelt die kategorialen Codes als Zahlen — das ist zwar mathematisch fragwuerdig, aber auf neuen Gemeinden schadet es weniger als bei Baummodellen
3. **Kleinste Generalisierungsluecke**: Ridge_D5 zeigt nur +0.2s Unterschied zwischen bekannten und unbekannten Gemeinden

**Laufzeit**: ca. 2 Minuten

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

SCRIPT_START = time.time()

In [ ]:
SCRIPT_DIR = Path(".").resolve().parent
DATA_DIR = SCRIPT_DIR / "data"
MODEL_DIR = SCRIPT_DIR / "models"
OUTPUT_DIR = SCRIPT_DIR / "ergebnisse"
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "travel_time"
VARIANTEN = ["D1_single", "D2_multi4", "D3_mittel6", "D4_gross10", "D5_fremd"]

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error."""
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

## Training

### Warum StandardScaler?
Ridge Regression bestraft alle Koeffizienten gleichmaessig mit $\alpha \|w\|^2$. Ohne Skalierung wuerden Features mit groesserer Skala (z.B. `segment_dist_m` in Metern) staerker bestraft als Features mit kleiner Skala (z.B. `hour_sin` im Bereich [-1, 1]).

Der StandardScaler normiert alle Features auf Mittelwert=0 und Standardabweichung=1.

### Hyperparameter
- `alpha=1.0`: Standard-Regularisierungsstaerke (hoeher = staerkere Regularisierung)

In [ ]:
results = []

for variant in VARIANTEN:
    print(f"\n{'='*60}")
    print(f"  Ridge | {variant}")
    print(f"{'='*60}")

    t0 = time.time()
    d = DATA_DIR / variant
    train = pd.read_csv(d / "train.csv")
    test = pd.read_csv(d / "test.csv")
    features = [c for c in train.columns if c != TARGET]

    # Skalierung: Zwingend noetig fuer Ridge
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[features].values)
    X_test = scaler.transform(test[features].values)
    y_train = train[TARGET].values
    y_test = test[TARGET].values

    print(f"  Train: {len(train):,}  Test: {len(test):,}  Features: {len(features)}")

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae_test = mean_absolute_error(y_test, y_pred)
    rmse_test = root_mean_squared_error(y_test, y_pred)
    mape_test = mape(y_test, y_pred)

    elapsed = time.time() - t0
    print(f"  Test MAE: {mae_test:.2f}s  RMSE: {rmse_test:.2f}s  MAPE: {mape_test:.1f}%")
    print(f"  Dauer: {elapsed:.1f}s")

    # Modell + Scaler speichern
    joblib.dump(model, MODEL_DIR / f"ridge_{variant}.joblib")
    joblib.dump(scaler, MODEL_DIR / f"ridge_scaler_{variant}.joblib")

    results.append({
        "experiment": f"Ridge_{variant}",
        "modell": "Ridge",
        "daten": variant,
        "train_n": len(train),
        "mae_test": round(mae_test, 2),
        "rmse_test": round(rmse_test, 2),
        "mape_test": round(mape_test, 1),
        "zeit_s": round(elapsed, 1),
    })

## Zusammenfassung

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "ridge_results.csv", index=False)
results_df[["experiment", "mae_test", "rmse_test", "mape_test", "zeit_s"]]

In [ ]:
print(f"Gesamtlaufzeit: {time.time() - SCRIPT_START:.1f}s")